# 🧠 3D VisReg JEPA Self-Supervised Pre-training on NVIDIA A100
### BraTS 2024 Adult Glioma Volumetric Pre-training Suite (Google Colab)

This notebook executes self-supervised 3D VisReg JEPA pre-training on **Google Colab with an A100 GPU**.

**Key Features & Parity with Kaggle Suite**:
1. **Full Algorithmic Parity**: Matches `notebooks/01_train_visreg_3d.ipynb` with explicit, modular CLI argument control.
2. **High-Resolution $8^3$ Patch Size**: Enables fine-grained voxel tokenization ($16^3 = 4,096$ tokens vs standard $8^3 = 512$ tokens) leveraging A100 40GB/80GB VRAM.
3. **Option B (3D Anisotropic Target Masking Jitter)**: Auto-scales across 6 multi-aspect ratio target cuboids (776 target tokens, 1,536 context tokens at 37.5%).
4. **Dual Cosine Schedulers**: Cosine learning rate decay with linear warmup ($5 \times 10^{-4} \to 1 \times 10^{-5}$) paired with cosine regularizer annealing ($1.0 \to 0.10$ after 15 warmup epochs).
5. **A100 Throughput**: Optimized `batch_size=16`, AMP FP16, and multi-worker CPU prefetching.
6. **Google Drive Checkpoint Streaming**: Persists checkpoints and logs directly to `/content/drive/MyDrive/checkpoints/` so no work is lost on disconnection.


## 1. Hardware & CUDA Environment Verification


In [ ]:
import datetime
import time
import torch

NOTEBOOK_START_TIME = time.time()
NOTEBOOK_START_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"⏱️ Session Start Time: {NOTEBOOK_START_STR}")

!nvidia-smi

print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Device Name:      {gpu_name}")
    print(f"Total VRAM:       {vram_gb:.2f} GB")
    if "A100" not in gpu_name:
        print("⚠️ Note: Optimal throughput is achieved on an A100 GPU (Runtime -> Change runtime type -> A100 GPU).")
    else:
        print("✓ A100 GPU confirmed. Ready for high-throughput 3D volumetric training.")
else:
    raise RuntimeError("No GPU detected! Go to Runtime -> Change runtime type -> Hardware accelerator -> GPU.")


## 2. Mount Google Drive (Checkpoint Safety)
Mounting Google Drive ensures that model checkpoints, training logs, and exported weights are saved permanently and survive session disconnects.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
output_dir = Path("/content/drive/MyDrive/checkpoints/visreg_jepa_a100")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"✓ Checkpoints and logs will stream directly to Google Drive:\n  {output_dir}")


## 3. Experiment Configuration & Reproducibility Parameters


In [ ]:
# =========================================================================
# ⚙️ Experiment Configuration & Reproducibility Parameters
# =========================================================================

SEED = 42                 # Deterministic seed for weight init, augmentations, and data splits
NUM_WORKERS = 4          # Parallel CPU data loading workers (Colab provides 8 vCPUs with A100)
PATCH_SIZE = 8           # High-resolution 8³ voxel patch size for A100 (4,096 token grid; default 16 for standard)
PRETRAIN_BATCH_SIZE = 16  # High-throughput volumetric batch size for A100
EPOCHS = 50              # Pre-training epochs (50 or 100)
LEARNING_RATE = 5e-4     # Base learning rate with linear warmup + cosine decay
WEIGHT_DECAY = 0.05      # AdamW weight decay


## 4. Dependencies Installation


In [ ]:
!pip install -q --no-cache-dir monai nibabel tabulate matplotlib
import monai
import nibabel as nib
import tabulate

print(f"✓ MONAI Version:    v{monai.__version__}")
print(f"✓ NiBabel Version:  v{nib.__version__}")
print(f"✓ Tabulate Version: v{tabulate.__version__}")


## 5. Codebase Setup & Editable Installation


In [ ]:
import os
import shutil
import sys
from pathlib import Path

REPO_URL = "https://github.com/hanriman/tumor_segmentation_3d.git"
work_dir = Path("/content/tumor_segmentation_3d")

# Clean up incomplete clone if needed
if work_dir.exists() and not (work_dir / "src" / "brats_jepa_3d").exists():
    print("⚠️ Cleaning up incomplete repository clone...")
    shutil.rmtree(work_dir)

if not (work_dir / "src" / "brats_jepa_3d").exists():
    print(f"Cloning codebase from: {REPO_URL} ...")
    !git clone {REPO_URL} {work_dir}

os.chdir(str(work_dir))
%cd {work_dir}

src_dir = work_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

!pip install -q -e .

print(f"\n✓ Working directory set to: {Path.cwd()}")
!git log -1 --oneline 2>/dev/null || echo "(Git commit info unavailable)"

!git -C {work_dir} checkout main
!git -C {work_dir} pull --ff-only origin main
assert (Path(work_dir) / "src/brats_jepa_3d/metrics/regions.py").exists(), \
    "Branch checkout failed: regions.py missing — aborting."
print("On branch main:", end=" ")
!git -C {work_dir} log -1 --oneline


## 6. Dataset Download & Discovery
Downloads the preprocessed BraTS 3D dataset into Colab fast local NVMe storage (`/content/data/`) and validates the data integrity.

> **Authentication**: Add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab Secrets (🔑 left sidebar), or upload your `kaggle.json`.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import torch
from brats_jepa_3d.config import get_dataset_dir, get_metadata_path
from brats_jepa_3d.data import BraTS3DDataset

KAGGLE_DATASET = "hanriman/brats-3d-full"  # Update if dataset slug differs

# 1. Authenticate Kaggle API
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("✓ Authenticated Kaggle API via Colab Secrets.")
except Exception:
    pass

if not os.environ.get("KAGGLE_KEY"):
    from google.colab import files
    print("Colab Secret not found. Please upload your kaggle.json:")
    files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    print("✓ Authenticated Kaggle API via kaggle.json.")

# 2. Fast download & unzip into local storage if not already present
if not Path("/content/data/metadata.csv").exists() and not any(Path("/content/data").glob("**/metadata.csv")):
    !mkdir -p /content/data
    !kaggle datasets download -d {KAGGLE_DATASET} -p /content/data --unzip

# 3. Discover dataset
data_dir = get_dataset_dir("brats_gli_3d_full")
meta_path = get_metadata_path("brats_gli_3d_full")
print(f"\n=== DATASET DISCOVERY ===")
print(f"Dataset Directory: {data_dir} (Exists: {data_dir.exists()})")
print(f"Metadata CSV:      {meta_path} (Exists: {meta_path.exists()})")

if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f"\n✓ Loaded Metadata: {len(df)} total records across splits:")
    print(df["split"].value_counts().to_string())

    ds = BraTS3DDataset(data_dir=data_dir, split="train")
    sample = ds[0]
    print("\n✓ Sample Tensor Verification:")
    print(f"  Image Shape: {sample['image'].shape} (dtype: {sample['image'].dtype})")
    print(f"  Mask Shape:  {sample['mask'].shape} (dtype: {sample['mask'].dtype})")
    print(f"  Tumor Voxels: {int((sample['mask'] > 0).sum()):,}")
else:
    raise FileNotFoundError(f"Metadata CSV not found at {meta_path}. Please verify Kaggle dataset download.")


## 7. Pre-Flight Verification Smoke Test (Optional, 2 Minutes)
Verifies CUDA, AMP, masking transform, and loss functions on 2 mini-batches with `--patch_size {PATCH_SIZE}` before starting the full pre-training run.


In [ ]:
!python scripts/train_jepa_3d.py \
    --data_dir {data_dir} \
    --model_type visreg_jepa \
    --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} \
    --smoke_test \
    --reg_warmup_epochs 2 \
    --reg_min_scale 0.10


## 8. Full 3D VisReg JEPA Self-Supervised Pre-training (A100)
Executes full self-supervised 3D VisReg JEPA pre-training with explicit CLI arguments (matching Kaggle suite):
- **Patch size**: `--patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE}` ($8^3$ voxel patches, $4,096$ tokens)
- **Batch size 16** with automatic mixed precision (`--amp`)
- **Cosine regularizer annealing** (`reg_warmup_epochs=15`, `reg_min_scale=0.10`)
- Checkpoints and logs stream directly to Google Drive


In [ ]:
!python scripts/train_jepa_3d.py \
    --data_dir {data_dir} \
    --output_dir {output_dir} \
    --model_type visreg_jepa \
    --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} \
    --seed {SEED} \
    --epochs {EPOCHS} \
    --batch_size {PRETRAIN_BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --learning_rate {LEARNING_RATE} \
    --weight_decay {WEIGHT_DECAY} \
    --reg_warmup_epochs 15 \
    --reg_min_scale 0.10 \
    --amp


## 9. Pre-training Diagnostics: Brain-Aware Masking Check


In [ ]:
!python scripts/diagnose_visreg_3d.py --num_volumes 200


## 10. Export & Package Pre-training Artifacts to Google Drive
Packages pre-trained weights, logs, and diagnostic reports into a verified zip archive in Google Drive.


In [ ]:
import datetime
import time
from pathlib import Path
from brats_jepa_3d.utils import export_artifacts

# 1. Package checkpoint and logs directly to Google Drive
zip_dest = Path("/content/drive/MyDrive/checkpoints/visreg_pretrain_outputs_a100.zip")
export_res = export_artifacts(
    export_name="visreg_pretrain_outputs_a100",
    export_dir=Path("/content/export_visreg_pretrain"),
    zip_path=zip_dest,
    source_dir=output_dir,
    model_prefix="visreg",
    verbose=True,
)

# 2. Session Timing Report
NOTEBOOK_END_TIME = time.time()
NOTEBOOK_END_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
start_time = globals().get("NOTEBOOK_START_TIME", NOTEBOOK_END_TIME)
total_elapsed_sec = NOTEBOOK_END_TIME - start_time
hours, rem = divmod(total_elapsed_sec, 3600)
minutes, seconds = divmod(rem, 60)

print("\n" + "=" * 50)
print(f"⏱️ Session Start Time:   {globals().get('NOTEBOOK_START_STR', 'N/A')}")
print(f"⏱️ Session End Time:     {NOTEBOOK_END_STR}")
print(f"⏱️ Total Execution Time: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({total_elapsed_sec:.2f}s)")
print("=" * 50)
print(f"✓ Checkpoint and diagnostic package saved to Google Drive:\n  {zip_dest}")
